# Week 13: Testing — Fixtures, Mocking, and CI Health

Same logic as `check_test_health.py`. This notebook runs the same two checks CI runs (`ruff check .` and `pytest --cov`), plus a hands-on look at the three testing techniques this week introduces: a new testable `call_llm` (mocked with `httpx.MockTransport`), a fixture that composes another fixture (`sample_collection`), and parametrized tests.

No API key needed — everything this week runs locally, no real network call.

## Run the Same Checks CI Runs

In [ ]:
import subprocess

for name, command in [
    ("Ruff (lint)", ["ruff", "check", "."]),
    ("Pytest (with coverage)", ["pytest", "--cov=ai_finance_course", "--cov-report=term-missing"]),
]:
    result = subprocess.run(command, capture_output=True, text=True, check=False)
    status = "PASS" if result.returncode == 0 else "FAIL"
    print(f"{status}  {name}")
    if result.returncode != 0:
        print(result.stdout[-2000:])
        print(result.stderr[-2000:])

## Testing an HTTP Call Without the Real Network

In [ ]:
import httpx

from ai_finance_course.llm_client import call_llm


def fake_anthropic_handler(request: httpx.Request) -> httpx.Response:
    # This function stands in for Anthropic's real API — Week 5 §4.3's
    # httpx.MockTransport pattern, applied to the LLM call for the first time.
    return httpx.Response(200, json={"content": [{"type": "text", "text": "Mocked response!"}]})


result = call_llm(
    "What is 2+2?",
    api_key="fake-key",
    model="fake-model",
    transport=httpx.MockTransport(fake_anthropic_handler),
)
print(result)

## Parametrized Tests: One Property, Many Inputs

In [ ]:
from ai_finance_course.chunking import chunk_text

for chunk_size, overlap in [(10, 2), (20, 5), (50, 0), (100, 25)]:
    text = "x" * 237
    chunks = chunk_text(text, chunk_size=chunk_size, overlap=overlap)
    within_limit = all(len(c) <= chunk_size for c in chunks)
    has_last_char = text[-1] in chunks[-1]
    print(f"chunk_size={chunk_size:>3} overlap={overlap:>2}  within_limit={within_limit}  has_last_char={has_last_char}  num_chunks={len(chunks)}")

## A Fixture That Composes Another Fixture

In [ ]:
# tests/conftest.py's sample_collection fixture depends on both the
# built-in tmp_path fixture and this file's own keyword_stub_embedding_function
# fixture — demonstrated here by calling the same functions it wires together.
import tempfile

from ai_finance_course.vector_store import add_chunks, get_or_create_collection
from tests.conftest import KeywordStubEmbeddingFunction

with tempfile.TemporaryDirectory() as tmp:
    collection = get_or_create_collection(tmp, "passages", KeywordStubEmbeddingFunction())
    add_chunks(collection, [{"text": "AAPL revenue grew 8% this quarter.", "ticker": "AAPL", "chunk_index": 0}])
    print("Collection count:", collection.count())